# Fase 5 — Segmentasi Kalimat + Entity Linking

**Input :** `news_tiered.parquet` (Fase 4) + `news_clean.parquet` (Fase 3)
**Output:**
- `sentences.parquet` — 1 baris per kalimat: `sentence_id, article_id, sent_idx, text_raw (track T), text_k (track K), n_tokens`
- `sentence_bank.parquet` — 1 baris per **(kalimat × bank)**: `sentence_id, bank, link_type, weight, text, published_at, tier`

Dua track teks: **T** (transformer, nyaris mentah) & **K** (klasik, lowercase+stopword+stemming Sastrawi).
Linking: `explicit` / `inherited` (anafora di artikel 1-bank) / `group` (Himbara, weight 0.5).

> Butuh `Sastrawi`. Install: `pip install Sastrawi` (sudah di requirements-crawl.txt terbaru).

In [ ]:
import os
from pathlib import Path
import pandas as pd
import yaml

if Path('src').exists():
    ROOT = Path('.').resolve()
elif Path('../src').exists():
    os.chdir('..'); ROOT = Path('.').resolve()
import sys; sys.path.insert(0, str(ROOT))
from src.link import build_link_rules
from src.pipeline_sentences import build_sentences

INTERIM = ROOT / 'data/interim'
tiered = pd.read_parquet(INTERIM / 'news_tiered.parquet')
clean = pd.read_parquet(INTERIM / 'news_clean.parquet')
aliases = yaml.safe_load(open(ROOT / 'config/aliases.yaml'))
print('artikel ter-tier:', tiered.article_id.nunique())

In [ ]:
# Catatan: stemming Sastrawi agak lambat. Perkiraan ~5-15 menit utk data penuh.
# Set do_stem=False bila ingin cepat (track K tanpa stemming).
rules = build_link_rules(aliases)
sentences, sentence_bank = build_sentences(tiered, clean, rules, do_stem=True)

sentences.to_parquet(INTERIM / 'sentences.parquet', index=False)
sentence_bank.to_parquet(INTERIM / 'sentence_bank.parquet', index=False)
print('tersimpan: sentences.parquet &  sentence_bank.parquet')

In [ ]:
print('===== RINGKASAN =====')
print('total kalimat        :', len(sentences))
print('pasangan kalimat-bank:', len(sentence_bank))
print('rata kalimat/artikel :', round(len(sentences)/max(tiered.article_id.nunique(),1), 1))

print('\n===== DISTRIBUSI link_type =====')
print(sentence_bank.link_type.value_counts().to_string())

print('\n===== PASANGAN per BANK =====')
print(sentence_bank.bank.value_counts().to_string())

print('\n===== pasangan per TIER =====')
print(sentence_bank.tier.value_counts().to_string())

In [ ]:
# QC: track T vs K + contoh tiap link_type (audit manual ~200 pasang untuk presisi >=95%)
print('===== TRACK T vs K =====')
for _, r in sentences.head(3).iterrows():
    print('T:', r.text_raw[:85])
    print('K:', r.text_k[:85]); print('-'*40)

for lt in ['explicit', 'inherited', 'group']:
    sub = sentence_bank[sentence_bank.link_type == lt]
    print(f'\n===== {lt} ({len(sub)}) =====')
    for _, r in sub.head(3).iterrows():
        print(f'  [{r.bank}] {r.text[:70]}')

## Setelah selesai
Paste **RINGKASAN**, **DISTRIBUSI link_type**, **per BANK/TIER**, dan contoh **QC** ke chat.
Kita cek kualitas segmentasi & linking, lalu lanjut Fase 6 (BERTopic).